In [2]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

load_dotenv()

hf_token = os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    huggingfacehub_api_token=hf_token
)

model = ChatHuggingFace(llm=llm)

c:\Users\HP\Documents\Coding journeys\GenAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import requests

In [4]:
# Tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated
@tool
def getConversionFactor(baseCurrency:str, targetCurrency:str)->float:
    """This function fetches the currency conversion factor in the base currency and targetCurrency"""

    url = f'https://v6.exchangerate-api.com/v6/6a829f293bfd88cd5185ed8e/pair/{baseCurrency}/{targetCurrency}'

    response = requests.get(url)
    
    return response.json()

@tool
def convert(baseCurrency:int, conversionRate:Annotated[float, InjectedToolArg])->float:
    """
    given a currency conversion rate is given function calculates the target currency value from a given base currency value
    """
    return baseCurrency * conversionRate


In [5]:
getConversionFactor.invoke({'baseCurrency':'USD', 'targetCurrency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1779667201,
 'time_last_update_utc': 'Mon, 25 May 2026 00:00:01 +0000',
 'time_next_update_unix': 1779753601,
 'time_next_update_utc': 'Tue, 26 May 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.8242}

In [6]:
convert.invoke({'baseCurrency' : 10, 'conversionRate' : 95.8824})

958.8240000000001

In [7]:
# Tool binding
modelWithTools = model.bind_tools([getConversionFactor, convert])

In [8]:
messages = [HumanMessage('What is the conversion factor in USD and INR, and based on that can you convert 10 USD to INR')]

In [9]:
messages

[HumanMessage(content='What is the conversion factor in USD and INR, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={})]

In [10]:
aiMessage = modelWithTools.invoke(messages)

In [11]:
aiMessage

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"baseCurrency":"USD","targetCurrency":"INR"}', 'name': 'getConversionFactor', 'description': None}, 'id': 'call_rpri4gpvklh0v8mmij8w5gb0', 'type': 'function'}, {'function': {'arguments': '{"baseCurrency":10}', 'name': 'convert', 'description': None}, 'id': 'call_l8x9xlu4867h71hzi2xmc5ml', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 341, 'total_tokens': 392}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e60d4-b613-7bc0-8d83-7c8810f34fd4-0', tool_calls=[{'name': 'getConversionFactor', 'args': {'baseCurrency': 'USD', 'targetCurrency': 'INR'}, 'id': 'call_rpri4gpvklh0v8mmij8w5gb0', 'type': 'tool_call'}, {'name': 'convert', 'args': {'baseCurrency': 10}, 'id': 'call_l8x9xlu4867h71hzi2xmc5ml', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_toke

In [12]:
messages.append(aiMessage)
aiMessage.tool_calls

[{'name': 'getConversionFactor',
  'args': {'baseCurrency': 'USD', 'targetCurrency': 'INR'},
  'id': 'call_rpri4gpvklh0v8mmij8w5gb0',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'baseCurrency': 10},
  'id': 'call_l8x9xlu4867h71hzi2xmc5ml',
  'type': 'tool_call'}]

In [13]:
import json

for toolCall in aiMessage.tool_calls:
    # Execute the firt tool and get the conversion rate
    if toolCall['name'] == 'getConversionFactor':
        toolMessage1 = getConversionFactor.invoke(toolCall)
        # print(toolMessage1)
        # fetch this conversion rate
        conversionRate = json.loads(toolMessage1.content)['conversion_rate']
        # Append this tool meesage to message list
        messages.append(toolMessage1)
    # and using the conversion rate run second tool
    if toolCall['name'] == 'convert':
        # Fetch the current Argument
        toolCall['args']['conversionRate'] = conversionRate
        toolMessage2 = convert.invoke(toolCall)
        messages.append(toolMessage2)

In [14]:
messages

[HumanMessage(content='What is the conversion factor in USD and INR, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"baseCurrency":"USD","targetCurrency":"INR"}', 'name': 'getConversionFactor', 'description': None}, 'id': 'call_rpri4gpvklh0v8mmij8w5gb0', 'type': 'function'}, {'function': {'arguments': '{"baseCurrency":10}', 'name': 'convert', 'description': None}, 'id': 'call_l8x9xlu4867h71hzi2xmc5ml', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 341, 'total_tokens': 392}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e60d4-b613-7bc0-8d83-7c8810f34fd4-0', tool_calls=[{'name': 'getConversionFactor', 'args': {'baseCurrency': 'USD', 'targetCurrency': 'INR'}, 'id': 'call_rpri4gpvklh0v8mmij8w5gb0', 'type': 'tool_call'

In [16]:
modelWithTools.invoke(messages).content

'The conversion rate from USD to INR as of the last update is approximately 95.8242. \n\nTo convert 10 USD to INR, you would multiply 10 by 95.8242, which gives you approximately 958.24 INR. \n\nSo, 10 USD is roughly equivalent to 958.24 INR.'